# Training a DQN Agent
The other training examples use [RLlib](rllib_training.ipynb) with PPO. BSK-RL environments
are ordinary [Gymnasium](https://gymnasium.farama.org/) environments, so they also work with
hand-written training loops. This example trains a
[Deep Q-Network](https://www.nature.com/articles/nature14236) (DQN) in plain PyTorch, with no
RL library.

DQN fits BSK-RL well for two reasons:

* **Discrete actions.** Most satellite action specs (scan, charge, downlink, desaturate,
  image target *n*) give a ``Discrete`` action space, which is what DQN needs.
* **Off-policy learning.** Simulator steps are expensive. A replay buffer lets every
  transition be reused for many gradient updates.

## Discounting in a semi-MDP
BSK-RL steps do not all last the same amount of time: a scan here lasts 180 seconds and a
downlink lasts 60. As described in the [time-discounted GAE](time_discounted_gae.ipynb)
example, the discount should depend on the elapsed time, not the number of steps. For
one-step Q-learning, the target becomes

\begin{equation}
y = r + \gamma^{\Delta t} \, (1 - \text{terminated}) \, \max_{a'} Q_\text{target}(s', a')
\end{equation}

where $\Delta t$ is the step duration in seconds, read from ``info["d_ts"]``, and $\gamma$ is a
*per-second* discount factor. This is the same convention as
``TimeDiscountedGAEPPOTorchLearner``, with reward credited at the end of the step.

## Load Modules

In [ ]:
import random
from typing import ClassVar

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from Basilisk.architecture import bskLogging
from torch import nn

from bsk_rl import SatelliteTasking, act, data, obs, sats, scene
from bsk_rl.sim import dyn, fsw

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Define the Environment
The environment is the nadir-scanning and downlink mission from the
[RLlib example](rllib_training.ipynb), so the two examples can be compared directly. The
satellite collects data while keeping its data buffer, battery, and reaction wheel speeds
within limits. If any of these limits is violated, the satellite fails, the episode ends,
and a penalty is applied.

In [ ]:
class ScanningDownlinkDynModel(
    dyn.ContinuousImagingDynModel, dyn.GroundStationDynModel
):
    # Define some custom properties to be accessed in the state
    @property
    def instrument_pointing_error(self) -> float:
        r_BN_P_unit = self.r_BN_P / np.linalg.norm(self.r_BN_P)
        c_hat_P = self.satellite.fsw.c_hat_P
        return np.arccos(np.dot(-r_BN_P_unit, c_hat_P))

    @property
    def solar_pointing_error(self) -> float:
        a = (
            self.world.gravFactory.spiceObject.planetStateOutMsgs[self.world.sun_index]
            .read()
            .PositionVector
        )
        a_hat_N = a / np.linalg.norm(a)
        nHat_B = self.satellite.sat_args["nHat_B"]
        NB = np.transpose(self.BN)
        nHat_N = NB @ nHat_B
        return np.arccos(np.dot(nHat_N, a_hat_N))


class ScanningSatellite(sats.AccessSatellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.SatProperties(
            dict(prop="storage_level_fraction"),
            dict(prop="battery_charge_fraction"),
            dict(prop="wheel_speeds_fraction"),
            dict(prop="instrument_pointing_error", norm=np.pi),
            dict(prop="solar_pointing_error", norm=np.pi),
        ),
        obs.OpportunityProperties(
            dict(prop="opportunity_open", norm=5700),
            dict(prop="opportunity_close", norm=5700),
            type="ground_station",
            n_ahead_observe=1,
        ),
        obs.Eclipse(norm=5700),
        obs.Time(),
    ]
    action_spec: ClassVar[list[act.Action]] = [
        act.Scan(duration=180.0),
        act.Charge(duration=120.0),
        act.Downlink(duration=60.0),
        act.Desat(duration=60.0),
    ]
    dyn_type = ScanningDownlinkDynModel
    fsw_type = fsw.ContinuousImagingFSWModel

The satellite parameters and environment arguments match the RLlib example. Because
``terminate_on_time_limit=True``, reaching the end of the mission is a true terminal
state, so the agent does not bootstrap past it.

In [ ]:
sat = ScanningSatellite(
    "Scanner_1",
    sat_args=dict(
        # Data
        dataStorageCapacity=5000 * 8e6,  # bits
        storageInit=lambda: np.random.uniform(0.0, 0.8) * 5000 * 8e6,
        instrumentBaudRate=0.5 * 8e6,
        transmitterBaudRate=-50 * 8e6,
        # Power
        batteryStorageCapacity=200 * 3600,  # W*s
        storedCharge_Init=lambda: np.random.uniform(0.3, 1.0) * 200 * 3600,
        basePowerDraw=-10.0,  # W
        instrumentPowerDraw=-30.0,  # W
        transmitterPowerDraw=-25.0,  # W
        thrusterPowerDraw=-80.0,  # W
        panelArea=0.25,
        # Attitude
        imageAttErrorRequirement=0.1,
        imageRateErrorRequirement=0.1,
        disturbance_vector=lambda: np.random.normal(scale=0.0001, size=3),  # N*m
        maxWheelSpeed=6000.0,  # RPM
        wheelSpeeds=lambda: np.random.uniform(-3000, 3000, 3),
        desatAttitude="nadir",
    ),
)

duration = 5 * 5700.0  # About 5 orbits
env = SatelliteTasking(
    satellite=sat,
    scenario=scene.UniformNadirScanning(value_per_second=1 / duration),
    rewarder=data.ScanningTimeReward(),
    time_limit=duration,
    failure_penalty=-1.0,
    terminate_on_time_limit=True,
    log_level="WARNING",
)

obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print("Observation size:", obs_dim)
print("Number of actions:", n_actions)

The step duration is reported in the info dictionary on every step:

In [ ]:
observation, info = env.reset(seed=SEED)
for action in range(n_actions):
    observation, reward, terminated, truncated, info = env.step(action)
    print(f"Action {action}: d_ts = {info['d_ts']:.0f} s, reward = {reward:.4f}")

## Q-Network and Replay Buffer
The Q-network is a small multilayer perceptron that maps an observation to one Q-value per
action. The replay buffer stores the usual $(s, a, r, s', \text{terminated})$ tuple **plus
the step duration** $\Delta t$, which the time-discounted target needs.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden=(128, 128)):
        super().__init__()
        layers, in_dim = [], obs_dim
        for width in hidden:
            layers += [nn.Linear(in_dim, width), nn.ReLU()]
            in_dim = width
        layers.append(nn.Linear(in_dim, n_actions))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity: int, obs_dim: int):
        self.capacity = capacity
        self.index = 0
        self.size = 0
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.d_ts = np.zeros(capacity, dtype=np.float32)

    def add(self, obs, action, reward, next_obs, terminated, d_t):
        i = self.index
        self.obs[i] = obs
        self.actions[i] = action
        self.rewards[i] = reward
        self.next_obs[i] = next_obs
        self.terminated[i] = terminated
        self.d_ts[i] = d_t
        self.index = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        idx = np.random.randint(0, self.size, size=batch_size)
        return tuple(
            torch.as_tensor(x[idx])
            for x in (
                self.obs,
                self.actions,
                self.rewards,
                self.next_obs,
                self.terminated,
                self.d_ts,
            )
        )

## DQN Agent
The agent keeps an online network and a slowly updated target network
([Polyak averaging](https://arxiv.org/abs/1509.02971) with rate ``tau``). It uses the Huber
loss and gradient clipping for stability. The only change from textbook DQN is the
discount: ``gamma ** d_t`` replaces a constant ``gamma``.

With ``gamma=0.9995`` per second, a reward one orbit (about 5700 s) away is weighted by
roughly 0.06. That is enough lookahead for the agent to learn to charge and desaturate
*before* a failure occurs.

In [ ]:
class DQNAgent:
    def __init__(
        self,
        obs_dim: int,
        n_actions: int,
        gamma: float = 0.9995,  # per second
        lr: float = 2.5e-4,
        tau: float = 0.005,
        batch_size: int = 64,
        buffer_size: int = 50_000,
    ):
        self.n_actions = n_actions
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.q = QNetwork(obs_dim, n_actions)
        self.q_target = QNetwork(obs_dim, n_actions)
        self.q_target.load_state_dict(self.q.state_dict())
        self.optimizer = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size, obs_dim)

    def act(self, obs, epsilon: float = 0.0) -> int:
        if random.random() < epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            q_values = self.q(torch.as_tensor(obs, dtype=torch.float32))
        return int(q_values.argmax())

    def update(self) -> float:
        obs, actions, rewards, next_obs, terminated, d_ts = self.buffer.sample(
            self.batch_size
        )
        with torch.no_grad():
            next_q = self.q_target(next_obs).max(dim=1).values
            # Semi-MDP target: discount by the elapsed time, not the step count
            target = rewards + self.gamma**d_ts * (1.0 - terminated) * next_q
        q = self.q(obs).gather(1, actions.unsqueeze(1)).squeeze(1)
        loss = F.smooth_l1_loss(q, target)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 10.0)
        self.optimizer.step()

        with torch.no_grad():
            for p_target, p in zip(self.q_target.parameters(), self.q.parameters()):
                p_target.mul_(1.0 - self.tau).add_(self.tau * p)
        return loss.item()


agent = DQNAgent(obs_dim, n_actions)

## Training
The training loop is a standard epsilon-greedy loop through the Gymnasium API. Epsilon
decays linearly over the first half of training, and updates start once the buffer holds
``learning_starts`` transitions.

Simulating a step costs much more than a gradient update on a small network, so the agent
takes ``updates_per_step = 4`` gradient steps for every environment step. Getting more
learning out of each simulated transition is the main practical advantage of an off-policy
method here.

To keep the documentation build fast, the agent trains for only 20000 environment steps
(about 100 episodes, around ten minutes on a two-core machine). That is enough to see the agent learn to
avoid most failures, but far from convergence. For real results, train much longer and tune
the hyperparameters.

In [ ]:
total_steps = 20000
learning_starts = 1_000
updates_per_step = 4
epsilon_start, epsilon_end = 1.0, 0.05
epsilon_decay_steps = total_steps // 2

episode_returns, episode_failures = [], []
step, episode = 0, 0

while step < total_steps:
    observation, info = env.reset(seed=SEED + episode)
    episode += 1
    done, episode_return = False, 0.0

    while not done:
        fraction = min(1.0, step / epsilon_decay_steps)
        epsilon = epsilon_start + fraction * (epsilon_end - epsilon_start)
        action = agent.act(observation, epsilon)

        next_observation, reward, terminated, truncated, info = env.step(action)
        agent.buffer.add(
            observation, action, reward, next_observation, terminated, info["d_ts"]
        )
        if agent.buffer.size >= learning_starts:
            for _ in range(updates_per_step):
                agent.update()

        observation = next_observation
        episode_return += reward
        step += 1
        done = terminated or truncated

    episode_returns.append(episode_return)
    episode_failures.append(not env.unwrapped.satellite.is_alive())
    if episode % 10 == 0:
        print(
            f"Episode {episode:3d} | step {step:6d} | epsilon {epsilon:.2f} | "
            f"mean return (last 10) {np.mean(episode_returns[-10:]):+.3f} | "
            f"failure rate (last 10) {np.mean(episode_failures[-10:]):.0%}"
        )

The learning curve shows returns rising as the agent stops running out of battery or
saturating its reaction wheels. Failed episodes end early and incur the ``-1`` penalty.

In [ ]:
window = 10
moving_average = np.convolve(episode_returns, np.ones(window) / window, mode="valid")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(episode_returns, alpha=0.3, label="Episode return")
ax.plot(
    np.arange(window - 1, len(episode_returns)), moving_average, label="10-episode mean"
)
ax.axhline(0.0, color="k", linewidth=0.5)
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("DQN training on the scanning/downlink environment")
ax.legend()
plt.show()

## Evaluating the Policy
The greedy policy (no exploration) is compared with a random policy on episodes seeded
differently from training.

In [ ]:
def evaluate(policy, n_episodes=10, seed=1000):
    returns, failures = [], []
    for k in range(n_episodes):
        observation, _ = env.reset(seed=seed + k)
        done, episode_return = False, 0.0
        while not done:
            observation, reward, terminated, truncated, _ = env.step(
                policy(observation)
            )
            episode_return += reward
            done = terminated or truncated
        returns.append(episode_return)
        failures.append(not env.unwrapped.satellite.is_alive())
    return np.mean(returns), np.std(returns), np.mean(failures)


for name, policy in [
    ("DQN (greedy)", lambda o: agent.act(o)),
    ("Random", lambda o: env.action_space.sample()),
]:
    mean, std, failure_rate = evaluate(policy)
    print(
        f"{name:13s} return {mean:+.3f} +/- {std:.3f} | failure rate {failure_rate:.0%}"
    )

## Next Steps
* **Save the policy** with ``torch.save(agent.q.state_dict(), "dqn.pt")``. At deployment
  time, the greedy action is ``agent.q(obs).argmax()``.
* **Train longer**, and consider standard DQN extensions such as
  [Double DQN](https://arxiv.org/abs/1509.06461), prioritized replay, or n-step returns.
  For n-step returns, discount by the *sum* of the step durations.
* **Scale up throughput.** Steps are cheap to store but slow to simulate. For large runs,
  collect experience from several environment copies in parallel, as the
  [RLlib example](rllib_training.ipynb) does.